In [29]:
# ---------------------------
# ESG_App (Jupyter, ipywidgets)
# ---------------------------
%matplotlib inline
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

sns.set_style('whitegrid')

# ---------------------------
# ESG Model (minimal)
# ---------------------------
class ESGModel:
    def __init__(self, dataset_path=None):
        if dataset_path and os.path.exists(dataset_path):
            self.data = pd.read_csv(dataset_path)
        else:
            self.data = pd.DataFrame()

# ---------------------------
# Persistent state
# ---------------------------
if 'esg_state' not in globals():
    esg_state = {
        "logged_in": False,
        "user_type": None,
        "username": "",
        "investor_rules": {}  # sector -> required_years
    }

RULES_FILE = "investor_rules.json"
if os.path.exists(RULES_FILE):
    try:
        with open(RULES_FILE,"r") as f:
            loaded = json.load(f)
            if isinstance(loaded, dict):
                esg_state['investor_rules'] = {str(k): int(v) for k,v in loaded.items()}
    except Exception:
        pass

# ---------------------------
# Widgets: Login / Logout
# ---------------------------
username_widget = widgets.Text(description="Username:")
password_widget = widgets.Password(description="Password:")
user_type_widget = widgets.RadioButtons(options=["Investor","Company"], description="Login as:")
login_button = widgets.Button(description="Login", button_style='success')
logout_button = widgets.Button(description="Logout", button_style='warning')
main_output = widgets.Output()

# FIX LOGOUT: attach globally
logout_button.on_click(lambda b: do_logout())

def do_logout(b=None):
    esg_state['logged_in'] = False
    esg_state['user_type'] = None
    esg_state['username'] = ""
    with main_output:
        clear_output(wait=True)
        display_login()

def on_login(b):
    with main_output:
        clear_output(wait=True)
        if username_widget.value.strip() and password_widget.value.strip():
            esg_state['logged_in'] = True
            esg_state['user_type'] = user_type_widget.value
            esg_state['username'] = username_widget.value.strip()
            display(HTML(f"<h3>Welcome, {esg_state['username']} ({esg_state['user_type']})</h3>"))
            display(logout_button)
            if esg_state['user_type']=="Investor":
                show_investor_portal()
            else:
                show_company_portal()
        else:
            display(HTML("<b style='color:red'>Please enter username and password.</b>"))
            display_login()

login_button.on_click(on_login)

def display_login():
    display(HTML("<h3>Login</h3>"))
    display(user_type_widget, username_widget, password_widget, login_button)
    if esg_state['investor_rules']:
        display(HTML("<b>Loaded investor rules:</b>"))
        display(pd.DataFrame(list(esg_state['investor_rules'].items()), columns=['Sector','Required Years']))

# ---------------------------
# Investor portal
# ---------------------------
def show_investor_portal():
    display(HTML("<h4>Investor Portal — Set minimum green years per sector</h4>"))
    
    main_sectors = [
        "Energy","Technology","Finance","Healthcare","IT","Materials",
        "Communication Services","Consumer Discretionary","Consumer Staples",
        "Real Estate","Utilities","Industrials","Chemical"
    ]
    
    slider_widgets = {}
    for s in main_sectors:
        current = esg_state['investor_rules'].get(s, 4)
        slider_widgets[s] = widgets.IntSlider(
            value=current, min=1, max=10, step=1, description=s, continuous_update=False
        )
        display(slider_widgets[s])

    # Separate output widget for displaying saved rules
    saved_rules_out = widgets.Output()
    with saved_rules_out:
        if esg_state['investor_rules']:
            display(HTML("<b>All sectors with saved rules:</b>"))
            display(pd.DataFrame(list(esg_state['investor_rules'].items()), columns=['Sector','Required Years']))
    display(saved_rules_out)
    
    save_btn = widgets.Button(description="Save Investor Rules", button_style='primary')
    out = widgets.Output()

    def save_rules(b):
        for s, w in slider_widgets.items():
            esg_state['investor_rules'][s] = int(w.value)
        with open(RULES_FILE,"w") as f:
            json.dump(esg_state['investor_rules'], f, indent=2)
        with out:
            clear_output(wait=True)
            display(HTML("<b style='color:green'>Investor rules saved.</b>"))
            # update saved rules output
            with saved_rules_out:
                clear_output(wait=True)
                display(HTML("<b>All sectors with saved rules:</b>"))
                display(pd.DataFrame(list(esg_state['investor_rules'].items()), columns=['Sector','Required Years']))

    save_btn.on_click(save_rules)
    display(save_btn, out)


# ---------------------------
# Company portal
# ---------------------------
# ---------------------------
# Company portal (robust required-years lookup + reload rules)
# ---------------------------
def show_company_portal():
    display(HTML("<h4>Company Portal — Upload CSV to Analyze ESG</h4>"))

    # ensure we have the latest investor rules loaded
    if os.path.exists(RULES_FILE):
        try:
            with open(RULES_FILE,"r") as f:
                loaded = json.load(f)
                if isinstance(loaded, dict):
                    esg_state['investor_rules'] = {str(k): int(v) for k,v in loaded.items()}
        except:
            pass

    if not esg_state['investor_rules']:
        display(HTML("<b style='color:orange'>No investor rules found yet. Please ask an investor to set rules.</b>"))

    upload = widgets.FileUpload(accept='.csv', multiple=False)
    process_btn = widgets.Button(description="Process CSV", button_style='success')
    output = widgets.Output()
    display(upload, process_btn, output)

    def normalize_text(s):
        if s is None:
            return ""
        return str(s).strip().lower()

    def get_required_years_for_sector(sector_text):
        """
        Return required years from esg_state['investor_rules'] for a given sector text.
        Attempts (in order):
          1) direct key lookup
          2) case-insensitive exact match
          3) substring match (sector contains key or key contains sector)
          4) fallback default 4
        """
        # reload rules to use latest
        rules = esg_state.get('investor_rules', {})
        if not rules:
            return 4
        # direct lookup
        if sector_text in rules:
            return int(rules[sector_text])
        # normalized lookup
        sector_norm = normalize_text(sector_text)
        # exact normalized match
        for k,v in rules.items():
            if normalize_text(k) == sector_norm:
                return int(v)
        # substring match (sector contains key or key contains sector)
        for k,v in rules.items():
            kn = normalize_text(k)
            if kn and kn in sector_norm:
                return int(v)
        for k,v in rules.items():
            kn = normalize_text(k)
            if sector_norm and sector_norm in kn:
                return int(v)
        # nothing matched — return default
        return 4

    def process_uploaded_csv(b):
        with output:
            clear_output(wait=True)
            if not upload.value:
                display(HTML("<b style='color:red'>Please upload a CSV file first.</b>"))
                return

            # handle ipywidgets value shape (tuple/list vs dict)
            if isinstance(upload.value, (tuple, list)):
                uploaded_file = upload.value[0]
                content = uploaded_file.get('content') or uploaded_file.get('data')
            elif isinstance(upload.value, dict):
                uploaded_file = list(upload.value.values())[0]
                content = uploaded_file['content']
            else:
                display(HTML("<b style='color:red'>Unsupported upload format.</b>"))
                return

            import io
            try:
                df = pd.read_csv(io.BytesIO(content))
            except Exception as e:
                display(HTML(f"<b style='color:red'>Error reading CSV: {e}</b>"))
                return

            # minimal column checks
            if 'Ticker' not in df.columns and 'Company' not in df.columns:
                display(HTML("<b style='color:red'>CSV must have 'Ticker' or 'Company' column.</b>"))
                return
            if 'Year' not in df.columns:
                display(HTML("<b style='color:red'>CSV must have 'Year' column.</b>"))
                return
            if 'Sector' not in df.columns:
                display(HTML("<b style='color:red'>CSV must have 'Sector' column.</b>"))
                return

            entity_col = 'Ticker' if 'Ticker' in df.columns else 'Company'

            # compute or find ESG_Score (unchanged logic)
            score_candidates = [c for c in df.columns if 'score' in c.lower() and 'esg' in c.lower()]
            if score_candidates:
                df['ESG_Score'] = pd.to_numeric(df[score_candidates[0]], errors='coerce')
            else:
                numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
                numeric_cols = [c for c in numeric_cols if c not in ['Year']]
                if not numeric_cols:
                    display(HTML("<b style='color:red'>Cannot compute ESG Score: no numeric columns found.</b>"))
                    return
                df['ESG_Score'] = df[numeric_cols].mean(axis=1)

            # ensure Year numeric
            df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')

            # Compute green years using threshold (50 — keeps previous behavior; you can expose slider)
            threshold = 50
            df['is_green'] = df['ESG_Score'] >= threshold

            # Important: normalize Sector text so we can match keys reliably later
            df['Sector_norm'] = df['Sector'].astype(str).str.strip()

            # Group by entity
            grouped = df.groupby(entity_col).agg({
                'Sector_norm': 'first',
                'is_green': 'sum',
                'Year': lambda x: x.nunique(),
                'ESG_Score': 'mean'
            }).rename(columns={'is_green':'Green_Years','Year':'Years_On_Record','ESG_Score':'Avg_ESG_Score','Sector_norm':'Sector'}).reset_index()

            # Reload rules one more time (in case changed recently)
            if os.path.exists(RULES_FILE):
                try:
                    with open(RULES_FILE,"r") as f:
                        loaded = json.load(f)
                        if isinstance(loaded, dict):
                            esg_state['investor_rules'] = {str(k): int(v) for k,v in loaded.items()}
                except:
                    pass

            # Map Required_Years using robust lookup
            grouped['Required_Years'] = grouped['Sector'].apply(get_required_years_for_sector)

            # Final status
            grouped['Final_Green_Status'] = grouped.apply(lambda r: 'Green' if r['Green_Years'] >= r['Required_Years'] else 'Not Green', axis=1)

            display(HTML("<b>Company ESG Report (with robust sector->required-years matching):</b>"))
            display(grouped[['Company' if entity_col=='Company' else 'Ticker','Sector','Years_On_Record','Green_Years','Required_Years','Avg_ESG_Score','Final_Green_Status']].rename(columns={entity_col: 'Company' if entity_col=='Company' else 'Ticker'}))

            # Plot
            plt.figure(figsize=(8,4))
            sns.histplot(grouped['Green_Years'], bins=range(0, int(grouped['Green_Years'].max())+2), kde=False)
            plt.title("Distribution of Green Years per Company")
            plt.xlabel("Green Years")
            plt.show()

    process_btn.on_click(process_uploaded_csv)

# End replacement function


# ---------------------------
# Start UI
# ---------------------------
with main_output:
    clear_output(wait=True)
    if esg_state['logged_in']:
        display(HTML(f"<h3>Welcome back, {esg_state['username']} ({esg_state['user_type']})</h3>"))
        display(logout_button)
        if esg_state['user_type']=="Investor":
            show_investor_portal()
        else:
            show_company_portal()
    else:
        display_login()

display(main_output)


Output()